In [2]:
!pip install -q -U google-genai pypdf

In [3]:
from google import genai
from google.colab import userdata
from google.genai import types
import numpy as np

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
EMB_MODEL="gemini-embedding-001"
EMB_DIM=768
MODEL="gemini-3.5-flash-lite"

In [6]:
#1.Prepare Document
#1.1 Upload document
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF:{pdf_name} is uploaded successfully")

Saving Customer_Support_RAG_Knowledge_Base.pdf to Customer_Support_RAG_Knowledge_Base.pdf
PDF:Customer_Support_RAG_Knowledge_Base.pdf is uploaded successfully


In [7]:
#1.2 Extract text from uploaded document
from pypdf import PdfReader
reader=PdfReader(pdf_name)
print("NUMBER OF PAGES: ",len(reader.pages))
text=""
for page in reader.pages:
  text+=page.extract_text()+"\n"
print(f"TEXT EXTRACTED FROM PDF HAVING {len(text)} CHARACTERS")


NUMBER OF PAGES:  1
TEXT EXTRACTED FROM PDF HAVING 3330 CHARACTERS


In [10]:
#1.3 Chunking
def chunk_text(text,chunk_size=800,overlap=15):
  start=0
  chunks=[]
  while start<len(text):
    end=start+chunk_size
    chunks.append(text[start:end])
    start=end-overlap
  return chunks

chunks=chunk_text(text)
print(f"CHUNKING DONE WITH {len(chunks)} CHUNKS")


CHUNKING DONE WITH 5 CHUNKS


In [11]:
#1.4 Embeddings
def embed_chunks(chunk):
  response=client.models.embed_content(
      model=EMB_MODEL,
      contents=chunk,
      config=types.EmbedContentConfig(
          output_dimensionality=EMB_DIM
      )
  )
  return response.embeddings[0].values

chunk_embed=[]
for chunk in chunks:
  chunk_embed.append(embed_chunks(chunks))
print("EMBEDDING DONE!")
chunk_embed=np.array(chunk_embed)
print("SHAPE: ",chunk_embed.shape)

EMBEDDING DONE!
SHAPE:  (5, 768)


In [13]:

def cosine_sim(a,b):
  a=np.array(a)
  b=np.array(b)

  return (float((np.dot(a,b))/(np.linalg.norm(a))*(np.linalg.norm(b))))


In [14]:
#Matching Issue and retrievig similar issues with their corresponding solutions.
def retrieve(issue,k=3):
  issue_embed=embed_chunks(issue)
  similarity=[]
  for i,chunk in enumerate(chunk_embed):
    score=cosine_sim(issue_embed,chunk)
    similarity.append((i,score))
  similarity.sort(key=lambda x:x[1],reverse=True)
  similarity=similarity[:k]
  return similarity


In [ ]:
#1.ANSWERING
system_instruction = f"""

Act as an AI Customer Support Assistant.

I will provide you with an issue that a user is facing.

Along with the issue, I will provide the top 3 most similar chunks retrieved from a knowledge-base PDF.
The PDF has already been processed by:
1. Extracting the text
2. Splitting the text into chunks
3. Generating embeddings
4. Performing similarity search

Your task is to answer the user's issue using ONLY the information available in the provided similar chunks.

Do not assume, invent, or add any information that is not available in the provided chunks.

If sufficient information to answer the issue is not available in the provided chunks, clearly state that the information is not available.

Return the response in JSON structured format.

The JSON must contain these keys:
- issue
- answer
- solution_steps
- escalation_required
- source

Always return valid JSON.
On every starting of new chatbot session reply Hi,I am your Customer Support Assistant.
"""
chats=client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=0.5,
        max_output_tokens=1800,
        system_instruction=system_instruction,
        response_mime_type="application/json",
        thinking_config=types.ThinkingConfig(thinking_level='low')
    ),
    history=[]
)
print("Enter your issue\nEnter exit quit bye to terminate session")
while True:
  issue_input=input("CUSTOMER: ")
  if issue_input.lower() in ['exit','bye','quit']:
    print("THANKYOU")
    break
  similarity=retrieve(issue_input)
  sim_chunk=[]
  for index,score in similarity:
    sim_chunk.append(chunks[index])

  prompt = f"""
User Issue:
{issue_input}

Retrieved Knowledge Base Chunks:
{sim_chunk}

Use ONLY the retrieved knowledge-base chunks above to answer the user's issue.

Do not use outside knowledge.
Do not assume missing information.
If the retrieved chunks do not contain enough information, state that the information is not available.

Return the answer in the required JSON format.
"""
  response=chats.send_message(prompt)
  data=json.loads(response.text)
  print(f"CUSTOMER ASSISTANT: {json.dumps(data,indent=3)}")

Enter your issue
Enter exit quit bye to terminate session
CUSTOMER: laptop is black heating
CUSTOMER ASSISTANT: {
  "issue": "laptop is black heating",
  "answer": "Hi, I am your Customer Support Assistant. For a black display, check the charging indicator, connect an external display, and perform the documented display reset procedure. For overheating, place the laptop on a hard, flat surface, keep ventilation openings clear, close unnecessary applications, and restart the device. If issues remain, contact technical support.",
  "solution_steps": [
    "Check the charging indicator",
    "Connect an external display",
    "Perform the documented display reset procedure",
    "Place the laptop on a hard, flat surface",
    "Keep ventilation openings clear",
    "Close unnecessary applications",
    "Restart the device"
  ],
  "escalation_required": true,
  "source": "Troubleshooting Guide"
}
CUSTOMER: wifi is not connecting
CUSTOMER ASSISTANT: {
  "issue": "wifi is not connecting",
  "